# Pipeline Huấn luyện & Đánh giá Tổng hợp Mô hình ResNet-18 (Bò, Gà, Lợn)

Notebook này gộp toàn bộ quy trình huấn luyện và đánh giá của cả **3 bài toán phân loại bệnh vật nuôi** (**Bò - Cow**, **Gà - Chicken**, **Lợn - Pig**) vào **1 file duy nhất** để chạy tự động từ đầu đến cuối chỉ trong 1 lần thực thi.

---

### Luồng thực thi chuẩn hóa (Unified Execution Flow):
1. **Khởi tạo & Cấu hình tổng thể**: Thiết lập môi trường, cố định seed ngẫu nhiên (`seed=42`), cấu hình siêu tham số (hyperparameters) và ánh xạ đường dẫn dữ liệu cho cả 3 loài.
2. **Kiểm tra dữ liệu trước huấn luyện (Dataset & Batch Sanity Check)**: Tự động kiểm tra cấu trúc thư mục `train/val/test`, thống kê phân bố mẫu, tính trọng số lớp (*Inverse Frequency Class Weights*) và kiểm tra kích thước tensor ảnh.
3. **Định nghĩa hàm Pipeline chuẩn hóa (`run_animal_pipeline`)**: Đóng gói quy trình khép kín cho mỗi loài:
   - **Phase 1 (Feature Extraction)**: Khóa backbone ResNet-18 pretrained ImageNet, chỉ huấn luyện lớp Classification Head mới (`10 epochs`, `lr=1e-3`, `AdamW` + `CosineAnnealingLR`).
   - **Đánh giá Phase 1**: Nạp lại checkpoint có Validation Accuracy cao nhất của Phase 1, đánh giá trên tập `test`, xuất báo cáo CSV/JSON, Confusion Matrix, Per-class F1 và Training Curves.
   - **Phase 2 (Full Fine-tuning)**: Mở khóa toàn bộ mạng ResNet-18, tinh chỉnh sâu toàn bộ trọng số với tốc độ học nhỏ hơn (`15 epochs`, `lr=1e-4`).
   - **Đánh giá Phase 2**: Nạp lại checkpoint tốt nhất của Phase 2, đánh giá toàn diện trên tập `test` và xuất bộ báo cáo/biểu đồ cuối cùng.
   - **Giải phóng tài nguyên (Memory Cleanup)**: Tự động dọn dẹp bộ nhớ RAM và GPU VRAM (`torch.cuda.empty_cache()`) sau mỗi loài để đảm bảo không tràn bộ nhớ khi chạy liên tiếp.
4. **Thực thi huấn luyện tuần tự (One-click Run)**: Chạy lần lượt qua danh sách các loài được chọn.
5. **Tổng kết & Đối chiếu hiệu năng toàn cục**: Lập bảng so sánh trực quan kết quả giữa Phase 1 và Phase 2 của tất cả các loài.

In [ ]:
import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# =============================================================================
# 1. XÁC ĐỊNH THƯ MỤC GỐC DỰ ÁN (PROJECT_ROOT) VÀ IMPORT MODULE TỪ SRC/
# =============================================================================
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset import load_dataset, make_loaders
from src.evaluate import plot_training_curves, run_full_evaluation
from src.model import build_model, count_parameters, load_checkpoint, unfreeze_backbone
from src.train import train_model
from src.utils import ANIMAL_DISPLAY_NAMES, get_pipeline_output_paths, set_seed

# =============================================================================
# 2. CẤU HÌNH DANH SÁCH LOÀI VẬT VÀ ĐƯỜNG DẪN DỮ LIỆU
# =============================================================================
# Ánh xạ tên loài sang thư mục dataset chuẩn cấu trúc ImageFolder (train/val/test)
ANIMAL_DATA_DIRS = {
    "cow": PROJECT_ROOT / "data" / "Cow_data",
    "chicken": PROJECT_ROOT / "data" / "Livestock_Dataset" / "Poultry_Feces",
    "pig": PROJECT_ROOT / "data" / "Livestock_Dataset" / "Livestock_Skin",
}

# Danh sách các loài sẽ được huấn luyện trong lần chạy này.
# - Mặc định: chạy tuần tự cả 3 loài ["cow", "chicken", "pig"] chỉ trong 1 lần bấm Run All.
# - Nếu chỉ muốn huấn luyện lại 1 loài cụ thể, bạn chỉ cần sửa thành ví dụ: ["pig"]
ANIMALS_TO_RUN = ["cow", "chicken", "pig"]

# =============================================================================
# 3. CẤU HÌNH SIÊU THAM SỐ HUẤN LUYỆN (HYPERPARAMETERS)
# =============================================================================
SEED = 42                 # Seed cố định để đảm bảo tính tái lập kết quả (Reproducibility)
BATCH_SIZE = 32           # Kích thước mini-batch (tối ưu cho GPU từ 4GB VRAM trở lên, giảm xuống 16 nếu thiếu VRAM)
PHASE1_EPOCHS = 10        # Số epoch huấn luyện Phase 1 (Feature Extraction - khóa backbone)
PHASE1_LR = 1e-3          # Learning rate cho Phase 1 (chỉ cập nhật lớp Linear phân loại mới)
PHASE2_EPOCHS = 15        # Số epoch huấn luyện Phase 2 (Full Fine-tuning - mở khóa toàn bộ mạng)
PHASE2_LR = 1e-4          # Learning rate cho Phase 2 (nhỏ hơn 10 lần để tinh chỉnh nhẹ nhàng trọng số ImageNet)

# Cố định random seed ngay từ đầu
set_seed(SEED)

# Kiểm tra thiết bị phần cứng (ưu tiên GPU CUDA nếu có)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 78)
print("CẤU HÌNH HỆ THỐNG HUẤN LUYỆN TỔNG HỢP (COW - CHICKEN - PIG)")
print("=" * 78)
print(f"Thư mục gốc dự án (PROJECT_ROOT) : {PROJECT_ROOT}")
print(f"Phiên bản PyTorch                : {torch.__version__}")
print(f"Thiết bị tính toán (Device)      : {device}")
if device.type == "cuda":
    print(f"Tên GPU                          : {torch.cuda.get_device_name(0)}")
print(f"Danh sách loài sẽ huấn luyện     : {ANIMALS_TO_RUN}")
print("-" * 78)
print("Kiểm tra trạng thái thư mục dữ liệu:")
for animal in ANIMALS_TO_RUN:
    data_path = ANIMAL_DATA_DIRS[animal]
    status = "[OK - Đã sẵn sàng]" if data_path.exists() else "[LỖI - Không tìm thấy thư mục]"
    print(f"  • {animal.upper():<8} : {data_path} -> {status}")
print("=" * 78)

In [ ]:
# =============================================================================
# KIỂM TRA DỮ LIỆU TỔNG QUAN (DATA SANITY CHECK) TRƯỚC KHI HUẤN LUYỆN
# =============================================================================
# Bước này kiểm tra toàn bộ các tập dữ liệu trong ANIMALS_TO_RUN ngay từ đầu
# để phát hiện sớm lỗi thiếu thư mục hoặc lệch nhãn trước khi chạy huấn luyện dài.

overview_rows = []

for animal in ANIMALS_TO_RUN:
    data_dir = ANIMAL_DATA_DIRS[animal]
    if not data_dir.exists():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục dữ liệu cho '{animal}' tại: {data_dir}\n"
            "Vui lòng kiểm tra và giải nén đầy đủ bộ dữ liệu trước khi chạy."
        )

    # Nạp thông tin dataset (train, val, test) và tự động tính trọng số lớp (class weights)
    train_ds, val_ds, test_ds, info = load_dataset(data_dir)

    overview_rows.append({
        "Loài (Animal)": animal.upper(),
        "Số lớp bệnh": info["n_classes"],
        "Train": f"{info['split_sizes']['train']:,}",
        "Val": f"{info['split_sizes']['val']:,}",
        "Test": f"{info['split_sizes']['test']:,}",
        "Tổng số ảnh": f"{info['split_sizes']['total']:,}",
    })

    print(f"\n>>> [{animal.upper()}] Danh sách {info['n_classes']} lớp bệnh và Trọng số cân bằng (Class Weights):")
    for idx, (cls_name, weight) in enumerate(zip(info["class_names"], info["class_weights"].tolist())):
        count = info["class_counts"][cls_name]
        print(f"  [{idx:02d}] {cls_name:<32} | Số ảnh Train: {count:>5,} | Trọng số (Weight): {weight:.4f}")

print("\n" + "=" * 78)
print("BẢNG TỔNG HỢP QUY MÔ DỮ LIỆU CỦA CÁC BÀI TOÁN")
print("=" * 78)
print(pd.DataFrame(overview_rows).to_string(index=False))
print("=" * 78)

In [ ]:
# =============================================================================
# ĐỊNH NGHĨA HÀM PIPELINE HUẤN LUYỆN & ĐÁNH GIÁ KHÉP KÍN CHO 1 LOÀI
# =============================================================================
def run_animal_pipeline(animal: str, data_dir: Path) -> dict:
    """
    Thực thi trọn vẹn quy trình huấn luyện 2 giai đoạn (Phase 1 & Phase 2)
    và đánh giá trên tập Test cho một loài vật cụ thể.

    Tham số:
    - animal: Tên định danh loài ('cow', 'chicken', hoặc 'pig').
    - data_dir: Đường dẫn thư mục chứa 3 tập con train/val/test của loài đó.

    Trả về:
    - Dict tổng hợp kết quả của cả Phase 1 và Phase 2 để phục vụ thống kê cuối cùng.
    """
    print("\n" + "#" * 78)
    print(f"# BẮT ĐẦU PIPELINE HUẤN LUYỆN & ĐÁNH GIÁ: {animal.upper()}")
    print("#" * 78)

    # 1. Đặt lại seed ngẫu nhiên trước mỗi loài để kết quả hoàn toàn độc lập và tái lập 100%
    set_seed(SEED)

    # 2. Lấy bộ đường dẫn xuất file chuẩn hóa cho cả Phase 1 và Phase 2
    paths_p1 = get_pipeline_output_paths(PROJECT_ROOT, animal=animal, phase="phase1")
    paths_p2 = get_pipeline_output_paths(PROJECT_ROOT, animal=animal, phase="phase2")

    # 3. Đọc dataset và khởi tạo DataLoaders
    train_dataset, val_dataset, test_dataset, info = load_dataset(data_dir)
    num_workers = 4 if torch.cuda.is_available() else 0

    train_loader, val_loader, test_loader = make_loaders(
        train_dataset,
        val_dataset,
        test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
    )

    # 4. Kiểm tra 1 mini-batch đầu vào & Khởi tạo mô hình ResNet-18 cho Phase 1
    sample_images, sample_labels = next(iter(train_loader))
    print(f"\n[1/6] Kiểm tra Batch & Khởi tạo Mô hình ({animal.upper()}):")
    print(f"  • Kích thước batch ảnh (B, C, H, W) : {sample_images.shape}")
    print(f"  • Dải giá trị pixel sau chuẩn hóa   : Min={sample_images.min().item():.3f}, Max={sample_images.max().item():.3f}")

    # Khởi tạo ResNet-18 pretrained ImageNet, đóng băng (freeze) toàn bộ phần trích xuất đặc trưng (backbone)
    model = build_model(
        n_classes=info["n_classes"],
        freeze_backbone=True,
    ).to(device)

    params_p1 = count_parameters(model)
    print(f"  • Tham số Phase 1 (Frozen Backbone) : Tổng={params_p1['total']:,} | Trainable={params_p1['trainable']:,} | Frozen={params_p1['frozen']:,}")

    # Chạy thử Forward Pass (Smoke Test) để đảm bảo kiến trúc và chiều đầu ra khớp số lớp
    with torch.no_grad():
        smoke_out = model(sample_images.to(device))
    print(f"  • Forward Smoke Test thành công! Output shape: {smoke_out.shape}")

    # =========================================================================
    # 5. PHASE 1: HUẤN LUYỆN FEATURE EXTRACTION (KHÓA BACKBONE)
    # =========================================================================
    print(f"\n[2/6] Huấn luyện Phase 1 - Feature Extraction ({PHASE1_EPOCHS} epochs, lr={PHASE1_LR})...")
    phase1_history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        n_epochs=PHASE1_EPOCHS,
        learning_rate=PHASE1_LR,
        class_weights=info["class_weights"].to(device),
        device=device,
        checkpoint_path=paths_p1["model_checkpoint"],
        history_path=paths_p1["history_json"],
        phase_name=f"phase1_frozen_backbone_{animal}",
        class_names=info["class_names"],
    )

    # =========================================================================
    # 6. ĐÁNH GIÁ PHASE 1 TRÊN TẬP TEST
    # =========================================================================
    # QUAN TRỌNG: Nạp lại checkpoint có Validation Accuracy cao nhất của Phase 1
    # (tránh dùng trọng số của epoch cuối nếu epoch cuối bị giảm hiệu năng)
    print(f"\n[3/6] Đánh giá Phase 1 trên tập Test (Checkpoint: {paths_p1['model_checkpoint'].name})...")
    load_checkpoint(model, paths_p1["model_checkpoint"], device=device)

    display_names = ANIMAL_DISPLAY_NAMES.get(animal, info["class_names"])

    phase1_report = run_full_evaluation(
        model=model,
        test_loader=test_loader,
        class_names=info["class_names"],
        device=device,
        results_dir=PROJECT_ROOT / "results",
        figures_dir=PROJECT_ROOT / "figures",
        label="phase1",
        display_names=display_names,
        prefix=f"{animal}_",
    )

    # Vẽ và lưu biểu đồ đường cong học tập (Loss & Accuracy) của Phase 1
    plot_training_curves(
        phase1_history,
        save_path=paths_p1["training_curves_png"],
    )

    # =========================================================================
    # 7. PHASE 2: MỞ KHÓA TOÀN BỘ MẠNG & FINE-TUNING TOÀN DIỆN
    # =========================================================================
    print(f"\n[4/6] Mở khóa toàn bộ Backbone ResNet-18 để chuyển sang Phase 2...")
    # Lưu ý: model hiện tại đang mang trọng số tốt nhất của Phase 1 (nhờ bước load_checkpoint ở trên)
    unfreeze_backbone(model)
    params_p2 = count_parameters(model)
    print(f"  • Tham số Phase 2 (Full Fine-tuning): Tổng={params_p2['total']:,} | Trainable={params_p2['trainable']:,} | Frozen={params_p2['frozen']:,}")

    print(f"\n[5/6] Huấn luyện Phase 2 - Full Fine-tuning ({PHASE2_EPOCHS} epochs, lr={PHASE2_LR})...")
    phase2_history = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        n_epochs=PHASE2_EPOCHS,
        learning_rate=PHASE2_LR,
        class_weights=info["class_weights"].to(device),
        device=device,
        checkpoint_path=paths_p2["model_checkpoint"],
        history_path=paths_p2["history_json"],
        phase_name=f"phase2_full_finetuning_{animal}",
        class_names=info["class_names"],
    )

    # =========================================================================
    # 8. ĐÁNH GIÁ PHASE 2 TRÊN TẬP TEST
    # =========================================================================
    print(f"\n[6/6] Đánh giá Phase 2 trên tập Test (Checkpoint: {paths_p2['model_checkpoint'].name})...")
    load_checkpoint(model, paths_p2["model_checkpoint"], device=device)

    phase2_report = run_full_evaluation(
        model=model,
        test_loader=test_loader,
        class_names=info["class_names"],
        device=device,
        results_dir=PROJECT_ROOT / "results",
        figures_dir=PROJECT_ROOT / "figures",
        label="phase2",
        display_names=display_names,
        prefix=f"{animal}_",
    )

    # Vẽ và lưu biểu đồ đường cong học tập (Loss & Accuracy) của Phase 2
    plot_training_curves(
        phase2_history,
        save_path=paths_p2["training_curves_png"],
    )

    # =========================================================================
    # 9. IN BẢNG TỔNG KẾT RIÊNG CHO LOÀI HIỆN TẠI
    # =========================================================================
    animal_summary_df = pd.DataFrame({
        "Chỉ số": ["Test Accuracy", "Macro F1", "Weighted F1", "Best Val Accuracy", "Learning Rate", "Epochs"],
        "Phase 1 (Frozen Backbone)": [
            f"{phase1_report['overall_accuracy'] * 100:.2f}%",
            f"{phase1_report['macro avg']['f1-score'] * 100:.2f}%",
            f"{phase1_report['weighted avg']['f1-score'] * 100:.2f}%",
            f"{phase1_history['best_val_accuracy'] * 100:.2f}%",
            f"{PHASE1_LR}",
            f"{PHASE1_EPOCHS}",
        ],
        "Phase 2 (Full Fine-tuning)": [
            f"{phase2_report['overall_accuracy'] * 100:.2f}%",
            f"{phase2_report['macro avg']['f1-score'] * 100:.2f}%",
            f"{phase2_report['weighted avg']['f1-score'] * 100:.2f}%",
            f"{phase2_history['best_val_accuracy'] * 100:.2f}%",
            f"{PHASE2_LR}",
            f"{PHASE2_EPOCHS}",
        ],
    })
    print("\n" + "-" * 70)
    print(f"TỔNG KẾT HIỆU NĂNG MÔ HÌNH: {animal.upper()}")
    print("-" * 70)
    print(animal_summary_df.to_string(index=False))
    print("-" * 70)

    # =========================================================================
    # 10. GIẢI PHÓNG BỘ NHỚ RAM & GPU VRAM TRƯỚC KHI CHUYỂN SANG LOÀI TIẾP THEO
    # =========================================================================
    del model, train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "animal": animal,
        "n_classes": info["n_classes"],
        "p1_val_acc": phase1_history["best_val_accuracy"],
        "p1_test_acc": phase1_report["overall_accuracy"],
        "p1_macro_f1": phase1_report["macro avg"]["f1-score"],
        "p1_weighted_f1": phase1_report["weighted avg"]["f1-score"],
        "p2_val_acc": phase2_history["best_val_accuracy"],
        "p2_test_acc": phase2_report["overall_accuracy"],
        "p2_macro_f1": phase2_report["macro avg"]["f1-score"],
        "p2_weighted_f1": phase2_report["weighted avg"]["f1-score"],
    }

In [ ]:
# =============================================================================
# THỰC THI HUẤN LUYỆN TUẦN TỰ CHO TẤT CẢ CÁC LOÀI TRONG 1 LẦN CHẠY
# =============================================================================
all_pipeline_results = []

for animal_name in ANIMALS_TO_RUN:
    dataset_dir = ANIMAL_DATA_DIRS[animal_name]
    res = run_animal_pipeline(animal=animal_name, data_dir=dataset_dir)
    all_pipeline_results.append(res)

print("\n" + "=" * 78)
print("HOÀN TẤT TOÀN BỘ QUá TRÌNH HUẤN LUYỆN VÀ ĐÁNH GIÁ CHO TẤT CẢ CÁC MÔ HÌNH!")
print("=" * 78)

In [ ]:
# =============================================================================
# BẢNG TỔNG KẾT & SO SÁNH HIỆU NĂNG TOÀN CỤC (COW - CHICKEN - PIG)
# =============================================================================
# Tổng hợp kết quả của tất cả các loài đã chạy để tiện đưa vào báo cáo / khóa luận

comparison_rows = []
for r in all_pipeline_results:
    acc_gain = (r["p2_test_acc"] - r["p1_test_acc"]) * 100
    f1_gain = (r["p2_macro_f1"] - r["p1_macro_f1"]) * 100

    comparison_rows.append({
        "Mô hình (Loài)": r["animal"].upper(),
        "Số lớp": r["n_classes"],
        "P1 Test Acc": f"{r['p1_test_acc'] * 100:.2f}%",
        "P2 Test Acc": f"{r['p2_test_acc'] * 100:.2f}%",
        "Tăng trưởng Acc": f"+{acc_gain:.2f}%",
        "P1 Macro F1": f"{r['p1_macro_f1'] * 100:.2f}%",
        "P2 Macro F1": f"{r['p2_macro_f1'] * 100:.2f}%",
        "P2 Weighted F1": f"{r['p2_weighted_f1'] * 100:.2f}%",
        "P2 Best Val Acc": f"{r['p2_val_acc'] * 100:.2f}%",
    })

master_summary_df = pd.DataFrame(comparison_rows)

print("\n" + "=" * 105)
print("BẢNG ĐỐI CHIẾU HIỆU NĂNG TỔNG HỢP TRÊN TẬP TEST (PHASE 1 vs PHASE 2)")
print("=" * 105)
print(master_summary_df.to_string(index=False))
print("=" * 105)